In [1]:
import numpy as np
import time
import os.path
import json
import matplotlib.pyplot as plt
import logging
logging.getLogger().setLevel(logging.INFO)

import xllim

## 1. Create Physical Model
    Select one physical model among TestModel, Hapke, Shkuratov, ExternalPythonModel

##### Test Model

In [2]:
model = xllim.TestModel()

##### Hapke

In [ ]:
# Get geometries from JSON file
with open("../dataRef/JSC1_BRDF.json", "r") as f:
    data = json.load(f)
geometries = np.array(data["JSC1_analogue"]["geometries"])

print("JSC1 geometries shape : {}".format(geometries.shape))


# Define other Hapke arguments
variant = "2002"
adapter = "six"
theta_bar_scaling = 30
b0 = 0
h = 0.1

# Create Hapke model
model = xllim.HapkeModel(geometries, variant, adapter, theta_bar_scaling, b0, h)

##### Shkuratov

In [ ]:
# Get geometries from JSON file
with open("../dataRef/JSC1_BRDF.json", "r") as f:
    data = json.load(f)
geometries = np.array(data["JSC1_analogue"]["geometries"])

print("JSC1 geometries shape : {}".format(geometries.shape))

# Define other Shkuratov arguments
variant = "5p"
scalingCoeffs = [1.0,1.5,1.5,1.5,1.5]
offset = [0,0,0.2,0,0]

# Create Shkuratov model
model = xllim.ShkuratovModel(geometries, variant, scalingCoeffs, offset)

##### External Python Model

In [5]:
# ExternalPythonModel(className, fileName, pathToFile)
model = xllim.ExternalPythonModel("ShkuratovModel5p", "ShkuratovModel5pPython", "../dataRef/externalPythonModels")

##### Check class infos and basic methods

In [ ]:
print(model.__doc__)
help(model)

In [ ]:
L = model.getDimensionX()
D = model.getDimensionY()
print("Test model transform a vector X of dimension L = {} into a vector Y of dimension D = {}\n".format(L, D))

x = np.ones(L)
x_physic = model.toPhysic(x)
x_mat = model.fromPhysic(x_physic)
print("model.toPhysic({}) = {}".format(x, x_physic))
print("model.fromPhysic({}) = {}\n".format(x_physic, x_mat))

x = np.ones(L)
y = model.F(x)
print("model.F({}) = ".format(x))
print(y)


## 2. dataGeneration

In [ ]:
N = 10_000                                          # number of generated observation
generator_type = "sobol"                            # the type of the generator used to generate x_gen matrix values
covariance = np.ones(model.getDimensionY()) * 1e-5  # vector of dimension D coresponding to the y_i variances.
# covariance = 0.1                                    # noise ratio float such as variances = y_i * nois_ratio
seed = 12345                                        # seed number for random generators

x_gen, y_gen = model.genData(N, generator_type, covariance, seed)

print(x_gen.shape)
print(y_gen.shape)

## 3. Train GLLiM model

In [9]:
# Choose Covariance matrix type
gamma_type = "full"
sigma_type = "diag"

# Define model dimensions
# ! D and L must fit the dataset dimensions
K = 5              # number of gaussians in the mixture model
D = y_gen.shape[1]  # = model.getDimensionY() = 9 (TestModel)
L = x_gen.shape[1]  # = model.getDimensionX() = 4 (TestModel)

# Hybrid model
n_hidden_variables = 0
L_t = L - n_hidden_variables
L_w = n_hidden_variables


gllim = xllim.GLLiM(K, D, L, gamma_type, sigma_type, n_hidden_variables)

##### Check theta

In [ ]:
print("Pi = {}".format(gllim.getParamPi()))

fake_Pi = np.zeros(K)
fake_Pi[0] = 1
gllim.setParamPi(fake_Pi)
print("Pi = {}".format(gllim.getParamPi()))

gllim.setParamPi(np.ones(K) / K)
print("Pi = {}".format(gllim.getParamPi()))

##### Initialisation

In [ ]:
# initialisation parameters
gllim_em_iteration = 10
gllim_em_floor = 1e-12
gmm_kmeans_iteration = 5
gmm_em_iteration = 10
gmm_floor = 1e-12
nb_experiences = 3

seed = 12345678
verbose = 1

# TODO : transpose in bindings & CARMA "can't be copied or stolen" : unable copying
x_gen_T = np.array(x_gen.T)
y_gen_T = np.array(y_gen.T)

gllim.initialize(
    x_gen_T,
    y_gen_T,
    gllim_em_iteration,
    gllim_em_floor,
    gmm_kmeans_iteration,
    gmm_em_iteration,
    gmm_floor,
    nb_experiences,
    seed,
    verbose,
)

##### Training

In [ ]:
# training parameters
train_max_iteration = 30
train_ratio_ll = 1e-5
train_floor = 1e-12

gllim.train(x_gen_T, y_gen_T, train_max_iteration, train_ratio_ll, train_floor, verbose)

##### Check theta_star, training insights

In [ ]:
theta_star = gllim.getInverse()
print("Pi* = {}".format(theta_star.Pi))

insights = gllim.getInsights()
print("Total training time = {} sec".format(insights.time))
print("Log-likelihood vector = {}".format(insights.log_likelihood))

## 4. Prediction

##### directDensities

In [ ]:
x = np.random.rand(L, 2)
x_incertitudes = np.random.rand(L) * 0.01

prediction_results = gllim.directDensities(x, x_incertitudes)
print("ŷ = GLLiM({})".format(x))
print("ŷ = {}".format(prediction_results.fullGMM.mean))
print("ŷ shape is {}".format(prediction_results.fullGMM.mean.shape))

##### inverseDensities

In [ ]:
y = np.random.rand(D, 1_000)
y_incertitudes = np.random.rand(D) * 0.01

tic = time.time()
prediction_results = gllim.inverseDensities(y, y_incertitudes)
print("Time One inversion for 1_000 observations = {} sec".format(time.time()-tic))

print("x_pred = GLLiM⁻¹({})".format(x[:,0]))
print("x_pred = {}".format(prediction_results.fullGMM.mean[0]))
print("x_pred shape is {}".format(prediction_results.fullGMM.mean.shape))

##### Regularize series

In [ ]:
series = prediction_results.mergedGMM.means
permutations = xllim.regularize(series)
print("permutations shape = {}".format(permutations.shape))
print(permutations)

##### JSC1 observations (physical model must correspond)

In [ ]:
# Get geometries from JSON file
with open("../dataRef/JSC1_BRDF.json", "r") as f:
    data = json.load(f)
wavelengths = np.array(data["JSC1_analogue"]["wavelengths"])
reflectances = np.array(data["JSC1_analogue"]["reflectance_JSC1"])
N_obs = wavelengths.shape[0]

print("There is {} observations !".format(N_obs))
print("wavelengths has shape : {}".format(wavelengths.shape))
print("reflectances has shape : {}".format(reflectances.shape))

observations = np.array(reflectances[:,0,:].T)
incertitudes = np.array(reflectances[:,1,:].T)

print("observations has shape : {}".format(observations.shape))
print("incertitudes has shape : {}".format(incertitudes.shape))


### Plot some Y parameters with respects to wavelengths ###
fig, axs = plt.subplots(1, 3, figsize=(30, 5))
fig.suptitle("Reflectances", fontsize=16)
y_idx_list = [3, 45, 68]
for d in range(len(y_idx_list)):
    axs[d].plot(wavelengths, observations[y_idx_list[d]], 'b-')
    axs[d].set_xlabel("Observations")
    axs[d].set_title('Y_'+ str(y_idx_list[d]))
    
plt.show()


In [ ]:
predictions = gllim.inverseDensities(observations, incertitudes, K_merged=2, merging_threshold=1e-10, verbose=0)

# comparing to y_obs
print("y_obs[0] = {}".format(observations[:,0]))
print("y_predicted[0] = F(x_predicted[0]) = {}".format(model.F(predictions.fullGMM.mean[0])))

## 5. Importance Sampling

##### Apply on fake proposition law

In [ ]:
K=5
# IMIS parameters
N_0, B, J = 10, 5, 2
covariance = np.ones(model.getDimensionY()) * 0.001 # Covariance on the target density

# Observations and incertitudes
N_obs = 100
y_obs = np.random.rand(N_obs, D)
y_err = y_obs * 0.001

# proposition laws (list of tuple(weights, means, covs))
proposition_gmms = []
for n in range(N_obs):
    weight = np.ones(K) * 1 / K
    mean = np.random.rand(L, K)
    cube = np.ones((L, L, K)) * 0.01
    cube += np.random.rand(L, L, K) * 0.1
    for k in range(cube.shape[2]):
        cube[:, :, k] += np.eye(L) * 0.1
        cube[:, :, k] = np.dot(cube[:, :, k], cube[:, :, k].T) * 0.001

    proposition_gmms.append((weight.T, mean, cube))

is_results = model.importanceSampling(proposition_gmms, y_obs, y_err, N_0, B, J, covariance, verbose=1)

##### Apply on GLLiM predictions

In [ ]:
# IMIS parameters
N_0, B, J = 10, 5, 2
covariance = np.ones(model.getDimensionY()) * 0.001 # Covariance on the target density


# TODO : transpose in bindings & CARMA "can't be copied or stolen" : unable copying
y_obs_noised_T = np.array(observations.T)
y_obs_noise_T = np.array(incertitudes.T)

# IMIS prediction by the mean
is_results = model.importanceSampling(predictions.mergedGMM, y_obs_noised_T, y_obs_noise_T, N_0, B, J, covariance, verbose=1)

# IMIS prediction by the centers
# perform importance sampling on a specific component of the merged GMM. i.e. a specific center.
idx_center = 0 # in [0, K_merged[
is_results_center_0 = model.importanceSampling(predictions.mergedGMM, y_obs_noised_T, y_obs_noise_T, N_0, B, J, verbose=1, covariance=covariance, idx_gaussian=idx_center)


## 6. Post-processing analysis

#### Evaluate reconstruction error

In [ ]:
def compute_reconstruction_error(reconstruction, observation):
    return np.linalg.norm(observation - reconstruction) / np.linalg.norm(observation)

reconstruction_error = []
reconstruction_error_is = []
N_obs = observations.shape[1]
for n in range(N_obs):
    reconstruction_error.append(
        compute_reconstruction_error(model.F(predictions.fullGMM.mean[n]), observations[:,n])
    )
    reconstruction_error_is.append(
        compute_reconstruction_error(model.F(is_results.predictions[:,n]), observations[:,n])
    )
    

plt.figure(figsize=(20, 5))
plt.plot(wavelengths, reconstruction_error, 'b*', label='GLLiM prediction')
plt.plot(wavelengths, reconstruction_error_is, 'g+', label='GLLiM + IS')
plt.xlabel("Observations")
plt.title("Reconstruction error for each observation")
plt.legend()
plt.grid()
    
plt.show()

#### Comparing estimation methods (GLLiM prediction and GLLiM + IS prediction)

In [ ]:
### Plot results ###
fig, axs = plt.subplots(2, L//2, figsize=(20, 10))
fig.suptitle("X Parameters estimations", fontsize=16)

for l in range(L):
    i = 0 if (l*2 < L) else 1
    j = l%(L//2)
    axs[i,j].plot(wavelengths, predictions.fullGMM.mean[:,l], 'b-', label='GLLiM prediction')
    axs[i,j].plot(wavelengths, is_results.predictions[l], 'g-', label='GLLiM + IS')
    axs[i,j].set_xlabel("Observations")
    axs[i,j].set_title('X_'+ str(l))
    axs[i,j].legend()
    
plt.show()